# threadlepy tutorial

This notebook introduces the main `threadlepy` wrappers in a runnable order. Related setup or workflow steps are grouped into the same cell; individual Threadle calls are still written clearly, usually one call per line, so each function usage is easy to see.

The final section mirrors the `th_attr_walk()` example in `threadleR/scripts/01_get_started.R`: it defines a small Python function that performs an attribute-filtered random walk by combining `get_random_node()`, `get_node_alters()`, and `get_attr()`.

You need a working Threadle CLI executable. This tutorial assumes `threadle` is available on `PATH`. If it is not, pass the full executable path to `th.start_threadle("/full/path/to/threadle")`.


## 1. Imports and startup

Start a Threadle JSON process for the notebook. In scripts, `with threadlepy.session():` is usually the cleaner lifecycle pattern.


In [1]:
import os
import random
from pathlib import Path

import threadlepy
import threadlepy.commands as th

threadlepy.__version__, threadlepy.is_available(), th.is_available()


('0.1.4', True, True)

In [2]:
threadlepy.configure(print_messages=False, timeout=1800)
th.start_threadle()

tutorial_workdir = th.get_workdir()
tutorial_workdir

threadle started pid=37819 pgid=37819


'/Users/doge/threadlepy/docs'

## 2. Working directory and example data

Use these helpers to inspect or set Threadle's working directory and to stage bundled example files locally.


In [3]:
th.get_workdir()
example_dir = th.stage_examples_to_wd("threadle_examples", overwrite=True)
th.set_workdir(example_dir)
th.dir()

{'Path': '/Users/doge/threadlepy/docs/threadle_examples',
 'Directories': [],
 'Files': [{'Name': 'dscw_nodeset.tsv'},
  {'Name': 'dscw.tsv'},
  {'Name': 'lazega_female_nodeset.tsv'},
  {'Name': 'lazega_female.tsv'},
  {'Name': 'lazega_nodes.tsv'},
  {'Name': 'lazega.tsv'},
  {'Name': 'mynet_nodesetfile.tsv'},
  {'Name': 'mynet.tsv'},
  {'Name': 'mynodes.tsv'}],
 'TotalDirectories': 0,
 'TotalFiles': 9}

In [4]:
th.sync_wd()
th.get_workdir()

'/Users/doge/threadlepy/docs'

`load_examples()` is the easiest way to load packaged data. The returned values are lightweight handles; the data live inside the Threadle backend session.


In [5]:
examples = th.load_examples(["mynet", "lazega"], assign=False)

mynet_nodeset = examples["mynet_nodeset"]
mynet = examples["mynet"]
lazega_nodeset = examples["lazega_nodeset"]
lazega = examples["lazega"]

In [6]:
loaded_nodes = th.load_file("loaded_nodes", os.path.join(example_dir, "mynet_nodesetfile.tsv"), type="nodeset")
loaded_net = th.load_file("loaded_net", os.path.join(example_dir, "mynet.tsv"), type="network")

## 3. Session metadata and low-level commands

`inventory()` and `i()` are equivalent. `cmd()` is a low-level escape hatch for Threadle commands that do not yet have a typed Python wrapper.


In [7]:
th.inventory()
th.i()
th.info(mynet)
th.preview(mynet)
th.cmd("info", args={"structure": mynet})


{'Type': 'Network',
 'Name': 'mynet',
 'Filepath': '/opt/miniconda3/lib/python3.13/site-packages/threadlepy/Examples/mynet.tsv',
 'isModified': False,
 'Nodeset': 'mynet_nodeset',
 'Layers': [{'Name': 'kinship',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Undirected',
   'ValueType': 'Binary',
   'SelftiesAllowed': False,
   'NbrEdges': 9,
   'EstimatedMemory': '856 B'},
  {'Name': 'trade',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': False,
   'NbrEdges': 9,
   'EstimatedMemory': '820 B'},
  {'Name': 'work',
   'Mode': 2,
   'Static': False,
   'NbrHyperedges': 4,
   'NbrAffiliations': 11,
   'EstimatedMemory': '1.4 kB'}],
 'EstimatedMemory': '4.3 kB'}

## 4. Create scratch structures

Use scratch structures for commands that mutate data.


In [8]:
scratch_nodes = th.create_nodeset("tutorial_nodes", name="Tutorial nodes", createnodes=5)
scratch_net = th.create_network("tutorial_net", scratch_nodes, name="Tutorial network")

th.info(scratch_nodes)
th.info(scratch_net)


{'Type': 'Network',
 'Name': 'Tutorial network',
 'Filepath': '',
 'isModified': False,
 'Nodeset': 'Tutorial nodes',
 'Layers': [],
 'EstimatedMemory': '92 B'}

## 5. Node functions

Node functions work on either a nodeset or a network.


In [9]:
th.get_nbr_nodes(scratch_nodes)
th.get_all_nodes(scratch_nodes, offset=0, limit=10)
th.get_nodeid_by_index(scratch_nodes, index=0)
th.get_random_node(scratch_nodes)


4

In [10]:
th.add_node(scratch_nodes, nodeid=10)
th.get_all_nodes(scratch_nodes, offset=0, limit=20)
th.remove_node(scratch_nodes, nodeid=10)
th.get_all_nodes(scratch_nodes, offset=0, limit=20)


[0, 1, 2, 3, 4]

## 6. Node attribute functions

Define an attribute, set values, read values, summarize them, and delete them. `generate_attr()` quickly creates random attribute values.


In [11]:
th.define_attr(scratch_nodes, "score", "int")
th.set_attr(scratch_nodes, nodeid=1, attrname="score", attrvalue=7)
th.set_attr(scratch_nodes, nodeid=2, attrname="score", attrvalue=9)

th.get_attr(scratch_nodes, nodeid=1, attrname="score")
th.get_attrs(scratch_nodes, nodes=[1, 2, 3], attrname="score")
th.get_attr_summary(scratch_nodes, attrname="score")


{'AttributeName': 'score',
 'AttributeType': 'Int',
 'Statistics': {'Mean': 8,
  'Median': 8,
  'StdDev': 1.4142135623730951,
  'Min': 7,
  'Max': 9,
  'Q1': 7.5,
  'Q3': 8.5,
  'Count': 2,
  'Missing': 3,
  'PercentageWithValue': 40}}

In [12]:
th.remove_attr(scratch_nodes, nodeid=1, attrname="score")
th.undefine_attr(scratch_nodes, "score")

th.generate_attr(scratch_nodes, "random_score", attrtype="int", min=1, max=10)
th.get_attr_summary(scratch_nodes, attrname="random_score")


{'AttributeName': 'random_score',
 'AttributeType': 'Int',
 'Statistics': {'Mean': 6.2,
  'Median': 8,
  'StdDev': 3.8987177379235853,
  'Min': 2,
  'Max': 10,
  'Q1': 2,
  'Q3': 9,
  'Count': 5,
  'Missing': 0,
  'PercentageWithValue': 100}}

In [13]:
th.get_attr(mynet_nodeset, nodeid=123, attrname="gender")
th.get_attrs(mynet_nodeset, nodes=[123, 234, 345], attrname="gender")
th.get_attr_summary(mynet_nodeset, attrname="gender")


{'AttributeName': 'gender',
 'AttributeType': 'Char',
 'Statistics': {'Frequency': {'m': 3, 'o': 1, 'f': 3},
  'Unique_Values': 3,
  'Mode': 'm',
  'Mode_Count': 3,
  'Count': 7,
  'Missing': 1,
  'PercentageWithValue': 87.5}}

## 7. One-mode layers and edge functions

Add a one-mode layer to the scratch network, then add, inspect, remove, and clear edges.


In [14]:
th.add_layer(scratch_net, "friendship", mode=1, directed=False, valuetype="binary", selfties=False)
th.add_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.add_edge(scratch_net, "friendship", node1id=2, node2id=3)

th.check_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.get_edge(scratch_net, "friendship", node1id=1, node2id=2)
th.get_all_edges(scratch_net, "friendship", offset=0, limit=10)
th.get_random_edge(scratch_net, "friendship")


{'node1': 2, 'node2': 1, 'value': 1}

In [15]:
th.remove_edge(scratch_net, "friendship", node1id=2, node2id=3)
th.clear_layer(scratch_net, "friendship")
th.get_all_edges(scratch_net, "friendship", offset=0, limit=10)


[]

## 8. Two-mode layers, hyperedges, and affiliations

A two-mode layer represents events, organizations, or categories as hyperedges. Affiliations connect nodes to those hyperedges.


In [16]:
th.add_layer(scratch_net, "events", mode=2)
th.add_hyper(scratch_net, "events", hypername="seminar", nodes=[1, 2])
th.add_aff(scratch_net, "events", nodeid=3, hypername="seminar")

th.get_all_hyperedges(scratch_net, "events", offset=0, limit=10)
th.get_hyperedge_nodes(scratch_net, "events", hypername="seminar")
th.get_node_hyperedges(scratch_net, "events", nodeid=3)


['seminar']

In [17]:
th.remove_aff(scratch_net, "events", nodeid=3, hypername="seminar")
th.remove_hyper(scratch_net, "events", hypername="seminar")


In [18]:
th.get_all_hyperedges(mynet, "work", offset=0, limit=10)
th.get_hyperedge_nodes(mynet, "work", hypername="ias")
th.get_node_hyperedges(mynet, "work", nodeid=123)


['ias']

## 9. Alters, paths, and network measures

These functions usually operate on an existing one-mode layer. `mynet` includes the one-mode layers `kinship` and `trade`.


In [19]:
th.get_edge(mynet, "trade", node1id=123, node2id=345)
th.check_edge(mynet, "kinship", node1id=123, node2id=345)
th.get_all_edges(mynet, "trade", offset=0, limit=10)
th.get_random_edge(mynet, "trade")


{'node1': 456, 'node2': 567, 'value': 23.45}

In [20]:
th.get_node_alters(mynet, nodeid=345, layernames="trade", direction="both")
th.get_degree(mynet, nodeid=345, layernames="trade", direction="both")
th.get_random_alter(mynet, nodeid=345, layernames="trade", direction="both")


456

In [21]:
th.shortest_path(mynet, node1id=123, node2id=567, layernames="kinship")
paths_by_gender = th.shortest_paths("paths_by_gender", mynet, attrname="gender", layernames="kinship")
th.info(paths_by_gender)


{'Type': 'Network',
 'Name': 'gender_sp_results',
 'Filepath': '',
 'isModified': False,
 'Nodeset': 'gender_values',
 'Layers': [{'Name': 'gender_sp_avg',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 14,
   'EstimatedMemory': '448 B'},
  {'Name': 'gender_sp_se',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 14,
   'EstimatedMemory': '448 B'},
  {'Name': 'gender_sp_count',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 14,
   'EstimatedMemory': '448 B'}],
 'EstimatedMemory': '1.9 kB'}

In [22]:
th.density(mynet, "kinship")
th.components(mynet, "kinship", attrname="kinship_component")
th.degree(mynet, "kinship", attrname="kinship_degree", direction="both")
th.get_attr_summary(mynet, attrname="kinship_degree")


{'AttributeName': 'kinship_degree',
 'AttributeType': 'Int',
 'Statistics': {'Mean': 2.25,
  'Median': 2,
  'StdDev': 1.164964745021435,
  'Min': 1,
  'Max': 4,
  'Q1': 1.75,
  'Q3': 2.5,
  'Count': 8,
  'Missing': 0,
  'PercentageWithValue': 100}}

## 10. Filtering, subnetworks, and layer transforms

These functions create new nodesets, networks, or layers from existing data.


In [23]:
female_nodeset = th.filter("female_nodeset", mynet_nodeset, attrname="gender", cond="eq", attrvalue="f")
female_nodeset_2 = th.filter_nodeset("female_nodeset_2", mynet_nodeset, attrname="gender", cond="eq", attrvalue="f")
female_mynet = th.subnet("female_mynet", mynet, female_nodeset)

th.info(female_mynet)


{'Type': 'Network',
 'Name': 'mynet_subnet',
 'Filepath': '',
 'isModified': True,
 'Nodeset': 'mynet_nodeset_clone',
 'Layers': [{'Name': 'kinship_filtered',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Undirected',
   'ValueType': 'Binary',
   'SelftiesAllowed': False,
   'NbrEdges': 0,
   'EstimatedMemory': '296 B'},
  {'Name': 'trade_filtered',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': False,
   'NbrEdges': 0,
   'EstimatedMemory': '296 B'},
  {'Name': 'work_filtered',
   'Mode': 2,
   'Static': False,
   'NbrHyperedges': 3,
   'NbrAffiliations': 4,
   'EstimatedMemory': '678 B'}],
 'EstimatedMemory': '1.8 kB'}

In [24]:
th.symmetrize(mynet, "trade", method="max", newlayername="trade_sym")
th.dichotomize(mynet, "trade", cond="ge", threshold=1000, truevalue=1, falsevalue=0, newlayername="trade_hi")
th.project_two_mode(mynet, "work", method="count", newlayername="work_projected")

th.get_edge(mynet, "trade_sym", node1id=123, node2id=345)
th.get_edge(mynet, "trade_hi", node1id=890, node2id=234)
th.get_edge(mynet, "work_projected", node1id=123, node2id=456)


1

In [25]:
th.pack(mynet, layername="kinship")
th.unpack(mynet, layername="kinship")


## 11. Random generation, settings, and seeds

Use these functions to configure backend behavior and generate random ties in a layer.


In [26]:
th.setting("verbose", False)
th.random_seed(42)

random_nodes = th.create_nodeset("random_nodes", createnodes=20)
random_net = th.create_network("random_net", random_nodes)
th.add_layer(random_net, "er_layer", mode=1, directed=False, valuetype="binary", selfties=False)
th.generate(random_net, "er_layer", type="er", p=0.15)
th.get_all_edges(random_net, "er_layer", offset=0, limit=10)


[{'node1': 0, 'node2': 3, 'value': 1},
 {'node1': 0, 'node2': 16, 'value': 1},
 {'node1': 1, 'node2': 11, 'value': 1},
 {'node1': 1, 'node2': 15, 'value': 1},
 {'node1': 2, 'node2': 9, 'value': 1},
 {'node1': 2, 'node2': 18, 'value': 1},
 {'node1': 3, 'node2': 4, 'value': 1},
 {'node1': 3, 'node2': 9, 'value': 1},
 {'node1': 4, 'node2': 5, 'value': 1},
 {'node1': 4, 'node2': 7, 'value': 1}]

## 12. Import, export, save, and scripts

This section writes a few temporary files, so it includes normal Python file handling around Threadle calls.


In [27]:
output_dir = Path(tutorial_workdir) / "threadle_tutorial_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

kinship_edges_file = output_dir / "mynet_kinship_edges.tsv"
mynet_gexf_file = output_dir / "mynet_kinship.gexf"
scratch_nodes_file = output_dir / "scratch_nodes_saved.tsv"
import_edges_file = output_dir / "import_edges.tsv"
script_file = output_dir / "threadle_script.txt"


In [28]:
th.export_layer(mynet, "kinship", file=str(kinship_edges_file), header=True, sep="\t")
kinship_edges_file.read_text(encoding="utf-8").splitlines()[:3]


['node1\tnode2', '123\t345', '345\t456']

In [29]:
th.export(mynet, file=str(mynet_gexf_file), layername="kinship", format="gexf")
th.save_file(scratch_nodes, file=str(scratch_nodes_file))

scratch_nodes_copy = th.load_file("scratch_nodes_copy", file=str(scratch_nodes_file), type="nodeset")
th.get_nbr_nodes(scratch_nodes_copy)


5

In [30]:
import_edges_file.write_text("1\t2\t4\n2\t3\t5\n", encoding="utf-8")

th.add_layer(scratch_net, "imported_edges", mode=1, directed=False, valuetype="valued", selfties=False)
th.import_layer(
    scratch_net,
    "imported_edges",
    file=str(import_edges_file),
    format="edgelist",
    node1col=0,
    node2col=1,
    valuecol=2,
    header=False,
    sep="\t",
    addmissingnodes=True,
)
th.get_edge(scratch_net, "imported_edges", node1id=1, node2id=2)


4

In [31]:
script_file.write_text(
    "script_nodes = createnodeset(name=script_nodes, createnodes=3)\n"
    "script_net = createnetwork(name=script_net, nodeset=script_nodes)\n"
    "addlayer(network=script_net, layername=script_layer, mode=1, directed=false, valuetype=binary, selfties=false)\n",
    encoding="utf-8",
)

th.load_script(str(script_file))
th.info("script_net")


{'Type': 'Network',
 'Name': 'script_net',
 'Filepath': '',
 'isModified': True,
 'Nodeset': 'script_nodes',
 'Layers': [{'Name': 'script_layer',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Undirected',
   'ValueType': 'Binary',
   'SelftiesAllowed': False,
   'NbrEdges': 0,
   'EstimatedMemory': '4 B'}],
 'EstimatedMemory': '64 B'}

## 13. Delete objects and layers

Deletion examples are best placed near the end of a tutorial.


In [32]:
th.add_layer(scratch_net, "temporary", mode=1)
th.remove_layer(scratch_net, "temporary")

delete_me = th.create_nodeset("delete_me", createnodes=1)
th.delete(delete_me)


## 14. Complete example: attribute-filtered random walk

This section follows the structure of `th_attr_walk()` in `threadleR/scripts/01_get_started.R`:

1. Choose a starting node; if none is supplied, use `get_random_node()`.
2. At each step, use `get_node_alters()` to find the current node's neighbors.
3. Use `get_attr()` to read each neighbor's attribute, then prefer neighbors whose attribute is in `keep`.
4. If no neighbor matches the attribute filter, use `fallback` to decide whether to stop or sample from all neighbors.
5. Return one row per step with `step`, `nodeid`, and the selected attribute value.


In [33]:
def attr_walk(
    network,
    start=None,
    steps=30,
    layernames=None,
    direction="both",
    attrname="gender",
    keep=("f",),
    unique=True,
    fallback="any",
):
    if fallback not in {"any", "stop"}:
        raise ValueError('fallback must be "any" or "stop"')

    cur = th.get_random_node(network) if start is None else start
    path = [cur]

    for _ in range(steps):
        alters = th.get_node_alters(
            network=network,
            nodeid=cur,
            layernames=layernames,
            direction=direction,
            unique=unique,
        )

        if not alters:
            if fallback == "stop":
                break
            path.append(None)
            break

        attr_values = {
            nodeid: str(th.get_attr(network, nodeid=nodeid, attrname=attrname))
            for nodeid in alters
        }
        candidates = [nodeid for nodeid in alters if attr_values[nodeid] in keep]

        if candidates:
            nxt = random.choice(candidates)
        elif fallback == "stop":
            break
        else:
            nxt = random.choice(alters)

        cur = nxt
        path.append(cur)

    return [
        {
            "step": step,
            "nodeid": nodeid,
            "attr": None if nodeid is None else th.get_attr(network, nodeid=nodeid, attrname=attrname),
        }
        for step, nodeid in enumerate(path)
    ]


In [34]:
lazega_nodeset = examples["lazega_nodeset"]
lazega = examples["lazega"]

th.info(lazega)
th.preview(lazega)


['Network: lazega',
 'Nodeset: Lawyers',
 ' friends [1-mode; Binary,Directed,False; Nbr edges:575]',
 '1 -> 2',
 '1 -> 4',
 '1 -> 8',
 '1 -> 17',
 '2 -> 16',
 '2 -> 17',
 '2 -> 22',
 '2 -> 26',
 '4 -> 2',
 '4 -> 3',
 ' advice [1-mode; Binary,Directed,False; Nbr edges:892]',
 '1 -> 2',
 '1 -> 17',
 '1 -> 20',
 '2 -> 1',
 '2 -> 6',
 '2 -> 17',
 '2 -> 20',
 '2 -> 22',
 '2 -> 24',
 '2 -> 26',
 ' collaboration [1-mode; Binary,Undirected,False; Nbr edges:378]',
 '1 <-> 17',
 '1 <-> 39',
 '1 <-> 40',
 '1 <-> 41',
 '17 <-> 19',
 '17 <-> 22',
 '17 <-> 24',
 '17 <-> 25',
 '17 <-> 26',
 '17 <-> 28']

In [35]:
walk1 = attr_walk(
    network=lazega,
    steps=20,
    layernames="friends",
    direction="both",
    attrname="Gender",
    keep=("M",),
    fallback="any",
)
walk1


[{'step': 0, 'nodeid': 4, 'attr': 'M'},
 {'step': 1, 'nodeid': 25, 'attr': 'M'},
 {'step': 2, 'nodeid': 14, 'attr': 'M'},
 {'step': 3, 'nodeid': 20, 'attr': 'M'},
 {'step': 4, 'nodeid': 17, 'attr': 'M'},
 {'step': 5, 'nodeid': 10, 'attr': 'M'},
 {'step': 6, 'nodeid': 8, 'attr': 'M'},
 {'step': 7, 'nodeid': 1, 'attr': 'M'},
 {'step': 8, 'nodeid': 8, 'attr': 'M'},
 {'step': 9, 'nodeid': 12, 'attr': 'M'},
 {'step': 10, 'nodeid': 22, 'attr': 'M'},
 {'step': 11, 'nodeid': 20, 'attr': 'M'},
 {'step': 12, 'nodeid': 26, 'attr': 'M'},
 {'step': 13, 'nodeid': 1, 'attr': 'M'},
 {'step': 14, 'nodeid': 12, 'attr': 'M'},
 {'step': 15, 'nodeid': 13, 'attr': 'M'},
 {'step': 16, 'nodeid': 41, 'attr': 'M'},
 {'step': 17, 'nodeid': 65, 'attr': 'M'},
 {'step': 18, 'nodeid': 56, 'attr': 'M'},
 {'step': 19, 'nodeid': 58, 'attr': 'M'},
 {'step': 20, 'nodeid': 35, 'attr': 'M'}]

In [36]:
try:
    import pandas as pd
    pd.DataFrame(walk1)
except ImportError:
    walk1


The cells below show Threadle's built-in random-walk summary commands. They differ from `attr_walk()`: `attr_walk()` returns one concrete path, while `rwdistances()` and `rwfpt()` return category-level result networks or statistics.


In [37]:
rw_distances = th.rwdistances(
    "rw_distances",
    lazega,
    attrname="Gender",
    maxsteps=30,
    layernames="friends",
    walkfactor=0.5,
    balanced=False,
    weighted=False,
    backtrack=False,
    savesteps=False,
)
th.info(rw_distances)


{'Type': 'Network',
 'Name': 'Gender_rwdistances_results',
 'Filepath': '',
 'isModified': False,
 'Nodeset': 'Gender_values',
 'Layers': [{'Name': 'Gender_avgdistance',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'},
  {'Name': 'Gender_stdevdistance',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'}],
 'EstimatedMemory': '738 B'}

In [38]:
rw_fpt = th.rwfpt(
    "rw_fpt",
    lazega,
    attrname="Gender",
    maxsteps=30,
    layernames="friends",
    walkfactor=0.5,
    minpairobs=1,
    balanced=False,
    weighted=False,
)
th.info(rw_fpt)


{'Type': 'Network',
 'Name': 'Gender_rwfpt_results',
 'Filepath': '',
 'isModified': False,
 'Nodeset': 'Gender_values',
 'Layers': [{'Name': 'Gender_fpt_avg',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'},
  {'Name': 'Gender_fpt_stdev',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'},
  {'Name': 'Gender_fpt_se',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'},
  {'Name': 'Gender_fpt_count',
   'Mode': 1,
   'Static': False,
   'Directionality': 'Directed',
   'ValueType': 'Valued',
   'SelftiesAllowed': True,
   'NbrEdges': 4,
   'EstimatedMemory': '212 B'},
  {'Name': 'Gender_fpt_coverage',
   'Mode': 1,
   'Static': False

In [39]:
rw_fpt_hist = th.rwfpt(
    "rw_fpt_hist",
    lazega,
    attrname="Gender",
    maxsteps=30,
    layernames="friends",
    walkfactor=0.5,
    minpairobs=1,
    balanced=False,
    weighted=False,
    return_histograms=True,
)
rw_fpt_hist


ThreadleStruct(name='rw_fpt_hist')

## 15. End the session

Clear backend objects if desired, then stop the background Threadle process.


In [40]:
th.delete_all()
th.i()
th.stop_threadle()


threadle stopped
threadle 37819 killed
